In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from utils import Conv, C3k2, SPPF, Bottleneck, Concat, C2PSA

import warnings
warnings.filterwarnings("ignore")

<h1><strong>Detect Block</strong></h1>

<h2><strong>Role of the Detect Class (YOLO11 Detection Head)</strong></h2>

The Detect class predicts both bounding box coordinates and class probabilities for each detection grid cell through:

- **<u>Multi-Scale Feature Processing:</u>**  
  Processes multi-scale feature maps using `cv2` and `cv3` convolutional layers, optimizing feature extraction for both object localization and class confidence estimation.

- **<u>Dual-Branch Architecture:</u>**  
  - **`cv2`**: Focuses on refining spatial features for accurate bounding box predictions, enhancing object localization.  
  - **`cv3`**: Primarily contributes to class confidence estimation, utilizing depthwise separable convolutions to improve computational efficiency.  

- **<u>Efficient and Accurate Localization:</u>**  
  - Leverages **Depthwise Convolutions (DWConv)** in `cv3` to reduce computational cost while preserving spatial information.
  - Integrates **Distribution Focal Loss (DFL)** to refine bounding box predictions, enhancing localization precision.

- **<u>Feature Fusion and Prediction:</u>**  
  Combines outputs from different feature scales, applying convolutional refinements and loss-based optimization to improve robustness and detection accuracy across varying object sizes.

By integrating convolutional processing, loss-based optimization, and feature fusion techniques, the `Detect` class enhances YOLO’s ability to accurately localize and classify objects, ensuring high detection performance with optimized efficiency.

In [2]:
class DWConv(Conv):
    """Depth-wise convolution"""
    def __init__(self, c1, c2, k=1, s=1, d=1, act=True):
        """Initialize depth-wise convolution with given parameters"""
        super().__init__(c1, c2, k, s, g=math.gcd(c1, c2), d=d, act=act)

In [3]:
class DFL(nn.Module):
    """
    Distribution Focal Loss (DFL) module.
    Converts raw anchor box predictions into actual box coordinates
    using a weighted sum over a discrete distribution of possible values.
    """
    def __init__(self, c1=16):
        super().__init__()
        self.conv = nn.Conv2d(c1, 1, 1, bias=False)  # 1x1 conv to collapse distribution
        x = torch.arange(c1, dtype=torch.float)
        self.conv.weight.data[:] = x.view(1, c1, 1, 1)
        self.c1 = c1

    def forward(self, x):
        b, c, a = x.shape  # batch, channels, anchors
        # reshape to [B, 4, reg_max, anchors] then softmax over reg_max dim
        return self.conv(x.view(b, 4, self.c1, a).transpose(2, 1).softmax(1)).view(b, 4, a)

In [4]:
def make_anchors(feats, strides, grid_cell_offset=0.5):
    """
    Generate anchor points for all feature map scales.
    Returns anchor center coordinates and corresponding strides.
    """
    anchor_points, stride_tensor = [], []
    dtype, device = feats[0].dtype, feats[0].device

    for i, stride in enumerate(strides):
        _, _, h, w = feats[i].shape
        # grid coordinates shifted by offset (0.5 = cell center)
        sx = torch.arange(w, device=device, dtype=dtype) + grid_cell_offset
        sy = torch.arange(h, device=device, dtype=dtype) + grid_cell_offset
        sy, sx = torch.meshgrid(sy, sx, indexing='ij')
        anchor_points.append(torch.stack((sx, sy), dim=-1).view(-1, 2))
        stride_tensor.append(torch.full((h * w, 1), stride, dtype=dtype, device=device))

    return torch.cat(anchor_points), torch.cat(stride_tensor)

In [5]:
def dist2bbox(distance, anchor_points, xywh=True, dim=-1):
    """
    Convert anchor-relative distance predictions to bounding boxes.

    distance : [lt, rb] — predicted distances from anchor to box edges
                (left, top, right, bottom)
    anchor_points : anchor center coordinates
    xywh : if True return (cx, cy, w, h), else (x1, y1, x2, y2)
    """
    lt, rb = distance.chunk(2, dim)               # left-top, right-bottom distances
    x1y1 = anchor_points - lt                     # top-left corner
    x2y2 = anchor_points + rb                     # bottom-right corner
    if xywh:
        c_xy = (x1y1 + x2y2) / 2                 # center x, y
        wh   = x2y2 - x1y1                        # width, height
        return torch.cat((c_xy, wh), dim)
    return torch.cat((x1y1, x2y2), dim)

In [6]:
class Detect(nn.Module):
    """YOLO Detect head for detection models"""

    end2end = False
    legacy = False

    def __init__(self, nc=80, ch=()):
        """
        Initializes the Detect head.

        Args:
            nc  : number of classes (e.g. 80 for COCO)
            ch  : tuple of input channels from neck, one per scale
                  e.g. (256, 512, 1024) for P3, P4, P5
        """
        super().__init__()
        self.nc = nc                        # number of classes
        self.nl = len(ch)                   # number of detection layers (3 for P3/P4/P5)
        self.reg_max = 16                   # DFL bins
        self.no = nc + self.reg_max * 4     # outputs per anchor = classes + 4*reg_max
        self.stride = torch.zeros(self.nl)  # filled during build
        self.shape = None
        self.anchors = torch.empty(0)
        self.strides = torch.empty(0)

        # channel sizes for bbox and cls branches
        c2 = max(16, ch[0] // 4, self.reg_max * 4)   # bbox branch hidden channels
        c3 = max(ch[0], min(self.nc, 100))            # cls branch hidden channels

        # ── bbox regression branch (cv2) ──────────────────────────────────────
        # one sub-network per scale: Conv→Conv→Conv2d → 4*reg_max channels
        self.cv2 = nn.ModuleList(
            nn.Sequential(
                Conv(x, c2, 3),
                Conv(c2, c2, 3),
                nn.Conv2d(c2, 4 * self.reg_max, 1)
            )
            for x in ch
        )

        # ── class prediction branch (cv3) ─────────────────────────────────────
        # legacy path = plain convs (v8 style)
        # default path = depthwise convs (v11 style, more efficient)
        self.cv3 = (
            nn.ModuleList(
                nn.Sequential(
                    Conv(x, c3, 3),
                    Conv(c3, c3, 3),
                    nn.Conv2d(c3, self.nc, 1)
                )
                for x in ch
            )
            if self.legacy
            else nn.ModuleList(
                nn.Sequential(
                    nn.Sequential(DWConv(x, x, 3), Conv(x, c3, 1)),     # depthwise + pointwise
                    nn.Sequential(DWConv(c3, c3, 3), Conv(c3, c3, 1)),  # depthwise + pointwise
                    nn.Conv2d(c3, self.nc, 1)                            # final class logits
                )
                for x in ch
            )
        )

        # DFL layer to decode bbox distributions into coordinates
        self.dfl = DFL(self.reg_max) if self.reg_max > 1 else nn.Identity()

    def forward(self, x):
        """
        Forward pass through all detection scales.

        Args:
            x : list of feature maps [P3, P4, P5]
                shapes e.g. [(B,256,80,80), (B,512,40,40), (B,1024,20,20)]

        Returns:
            During training : list of raw predictions per scale
            During inference: decoded (boxes, scores) tensor
        """
        for i in range(self.nl):
            # concatenate bbox and cls predictions along channel dim
            x[i] = torch.cat((self.cv2[i](x[i]), self.cv3[i](x[i])), dim=1)

        if self.training:
            return x  # raw predictions, loss computed externally

        # ── inference decoding ────────────────────────────────────────────────
        shape = x[0].shape  # B, C, H, W
        # flatten spatial dims and concat all scales → [B, no, total_anchors]
        x_cat = torch.cat([xi.view(shape[0], self.no, -1) for xi in x], dim=2)

        # recompute anchors only if input shape changed
        if self.shape != shape:
            anchors, strides = make_anchors(x, self.stride, 0.5)
            self.anchors = anchors.transpose(0, 1)   # [2, 8400]
            self.strides = strides.transpose(0, 1)   # [1, 8400]
            self.shape = shape

        # split into bbox distribution and class logits
        box, cls = x_cat.split((self.reg_max * 4, self.nc), dim=1)

        # decode DFL distribution → actual box (cx, cy, w, h)
        dbox = self.decode_bboxes(self.dfl(box), self.anchors.unsqueeze(0)) * self.strides

        # return [boxes + scores] concatenated
        return torch.cat((dbox, cls.sigmoid()), dim=1)

    def decode_bboxes(self, bboxes, anchors):
        """Decode predicted bbox deltas into (x1,y1,x2,y2) format"""
        return dist2bbox(bboxes, anchors, xywh=True, dim=1)

    def bias_init(self):
        """
        Initialize biases for better training stability.
        Sets cls bias based on prior object probability,
        sets bbox bias to reasonable starting values.
        """
        for a, b, s in zip(self.cv2, self.cv3, self.stride):
            # bbox bias: last Conv2d bias in cv2
            a[-1].bias.data[:] = 1.0
            # cls bias: initialise with log-prior so sigmoid(bias) ≈ 0.01
            b[-1].bias.data[:self.nc] = math.log(5 / self.nc / (640 / s) ** 2)